# Check attention in a previously trained model

Inspect the four **spatial attention gates** over one MRI patch. These are scalar
sigmoid weights applied to skip features, shared across classes; they are not
transformer token matrices, segmentation probabilities, or causal explanations.

Run cells in order: load a checkpoint, select an MRI, preprocess it, choose a
patch, extract gate coefficients, then explore slices with native Plotly.
No training, optimizer restoration, post-processing or source-file writes occur.
Models with attention disabled have no gate maps and report that explicitly.

In [ ]:
from pathlib import Path
from typing import Literal
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/models/networks.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import nibabel as nib
import numpy as np
from IPython.display import display
from src.evaluation.attention import (
    load_attention_model, attention_gate_table, inspect_attention, attention_figure,
)
from src.experiments.training import configure_runtime
from src.preprocessing.preprocessor import (
    CorrectX10, Normalize, PadderCutter, Preprocessor, RandomCropping, Reorient, Resample,
)
from src.utils.inspection import sample_normalized_scan

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
print("Reference axes:", nib.aff2axcodes(ref_img.affine))
runtime = configure_runtime(seed=42, memory_growth=True, mixed_precision=False)

## 1. Select the saved model

The default is your explicit MouseFull v1 run. `FOLD` selects a CV member. The
loader follows its saved **selected best checkpoint**, never the most recent file.

For a different weights file, set `MODEL_FILE` and its `EXPERIMENT_FILE`.
For an older complete export, set `RUN_DIR=None`, `CHECKPOINT_KIND="model"` and
`MODEL_FILE=REPO_ROOT / "Models" / "rnet_da.h5"` (or `rnet_mice.h5` / `rnet_rats.h5`).
Complete exports need no experiment file, but their preprocessing settings below
must match their training convention. Use trusted model files, as for existing
Keras/HDF5 loading elsewhere in the repository.

In [ ]:
RUN_DIR: Path | None = (
    REPO_ROOT / "results" / "mouse_full_3drnet" / "RUN_20261014_182511_289035"
)
FOLD: int = 0
MODEL_FILE: Path | None = None
EXPERIMENT_FILE: Path | None = None
CHECKPOINT_KIND: Literal["weights", "model"] = "weights"

model, saved_config, checkpoint_file = load_attention_model(
    run_dir=RUN_DIR, fold=FOLD, model_file=MODEL_FILE,
    experiment_file=EXPERIMENT_FILE, checkpoint_kind=CHECKPOINT_KIND,
)
print("Checkpoint:", checkpoint_file)
print("Model input:", model.input_shape)
gate_layers = attention_gate_table(model)
display(gate_layers)

## 2. Choose an MRI

Use a reproducibly sampled normalized animal (`D1`, `D2`, `D3` or `D4`), or
`INPUT_MODE="files"` with an explicit `IMAGE_PATH` and optional `ROI_PATH`.
Legacy MRI paths work in files mode too. ROI inputs use **original source IDs**.
The MRI should already have the N4 correction used for training; this notebook
does not add N4 correction. Change `SEED` to sample another animal.

In [ ]:
INPUT_MODE: Literal["normalized", "files"] = "normalized"
DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
DATASET: str = "D1"
SEED: int = 42
IMAGE_PATH: Path | None = None
ROI_PATH: Path | None = None

roi_image = None
if INPUT_MODE == "normalized":
    scan = sample_normalized_scan(root=DATA_ROOT, dataset=DATASET, seed=SEED)
    image, roi_image, _ = scan.get_images()
    print("Selected scan:", scan.id, scan.subject_group)
elif INPUT_MODE == "files":
    if IMAGE_PATH is None:
        raise ValueError("Set IMAGE_PATH for files mode")
    image = nib.load(IMAGE_PATH)
    if ROI_PATH is not None:
        roi_image = nib.load(ROI_PATH)
    print("Selected MRI:", IMAGE_PATH)
else:
    raise ValueError("INPUT_MODE must be normalized or files")

if image.ndim != 3:
    raise ValueError("Choose a 3D MRI volume")
if roi_image is not None:
    if roi_image.shape != image.shape or not np.allclose(roi_image.affine, image.affine):
        raise ValueError("The source ROI mask must share the MRI grid")
print("Source shape:", image.shape, "axes:", nib.aff2axcodes(image.affine))

## 3. Prepare the input, using the common RIA reference

Saved runs supply spacing, unit correction and normalization. The reference is
always the shared RIA reference requested for the notebooks, including old models.
For complete legacy models without an experiment file, edit the fallback settings
explicitly. MRI uses linear interpolation, ROI masks nearest neighbor. All spatial
operations use the same parameters for MRI and masks. Original GT IDs are retained
for display; no output-class mapping or post-processing is applied to GT.

In [ ]:
LEGACY_PATCH_SIZE: tuple[int, int, int] = (80, 80, 80)  # For exports with flexible input dimensions.
LEGACY_SPACING_MM: tuple[float, float, float] = (0.1, 0.1, 0.1)
LEGACY_CORRECT_UNITS: bool = True
LEGACY_NORMALIZATION: Literal["zscore", "minmax"] = "zscore"

spacing_mm = LEGACY_SPACING_MM
correct_units = LEGACY_CORRECT_UNITS
normalization = LEGACY_NORMALIZATION
if saved_config is not None:
    spacing_mm = saved_config.preprocessing.spacing_mm
    correct_units = saved_config.preprocessing.correct_units
    normalization = saved_config.preprocessing.normalization

steps = [Reorient(ref_img=ref_img)]
if correct_units:
    correction = CorrectX10(reference_resolution=spacing_mm)
    steps.append(correction)
resampling = Resample(target_resolution=spacing_mm, interpolation=1, orthogonal=False)
intensity_normalization = Normalize(mode=normalization)
steps.extend([resampling, intensity_normalization])
pipeline = Preprocessor(steps=steps)

masks = []
if roi_image is not None:
    masks.append(roi_image)
processed_roi = None
if masks:
    processed_image, processed_masks = pipeline.preprocess(imgs=image, masks=masks)
    processed_roi = processed_masks[0]
else:
    processed_image = pipeline.preprocess(imgs=image)
print("Processed shape:", processed_image.shape, "axes:", nib.aff2axcodes(processed_image.affine))
print("Spacing:", spacing_mm, "normalization:", normalization)

## 4. Choose the patch to inspect

`PATCH_CENTER=None` takes a deterministic center crop, padding smaller volumes.
Otherwise provide `(x, y, z)` voxel coordinates in the processed volume, choosing
a center that fits the full patch. This is **patch-level attention**, not an
aggregation over a sliding-window whole-brain prediction. Changing patch context
can change the attention gates.

In [ ]:
PATCH_CENTER: tuple[int, int, int] | None = None

patch_size = LEGACY_PATCH_SIZE
if saved_config is not None:
    patch_size = saved_config.preprocessing.patch_size
if all(size is not None for size in model.input_shape[1:4]):
    patch_size = tuple(int(size) for size in model.input_shape[1:4])
patch_roi = None
if PATCH_CENTER is None:
    crop = PadderCutter(img_size=patch_size)
    crop_pipeline = Preprocessor(steps=[crop])
    if processed_roi is None:
        patch_image = crop_pipeline.preprocess(imgs=processed_image)
    else:
        patch_image, patch_masks = crop_pipeline.preprocess(imgs=processed_image, masks=[processed_roi])
        patch_roi = patch_masks[0]
else:
    crop = RandomCropping(dim=patch_size, mode="center", std=None, padding=False)
    patch_image = crop.execute(image=processed_image, center=PATCH_CENTER)
    if processed_roi is not None:
        patch_roi = crop.execute(image=processed_roi, is_mask=True, center=PATCH_CENTER)
print("Model patch:", patch_image.shape)

## 5. Extract the gate coefficients

Inference uses `training=False` so dropout is off and batch-normalization
statistics are unchanged. The table describes the native coefficients before
channel repetition. Coarse maps are linearly lifted onto the patch grid only for
visualization. Their real sigmoid scale is preserved; maps are not individually
rescaled to look more contrasted. Low-resolution gates can appear blocky because
the network itself upsamples their weights before applying them to skip features.

In [ ]:
attention_result = inspect_attention(model=model, image=patch_image)
display(attention_result.summary)

## 6. Explore the maps

The slider uses the same patch slice in every panel; zoom and hover use native
Plotly, without widgets or HTML output. Color means **gate weight from 0 to 1**.
`GT_REGION=1` outlines the original lesion annotation in cyan when available;
`None` outlines all annotated foreground. Set `SHOW_GT=False` for MRI/maps only.
These gates are shared across segmentation classes and do not establish why a
particular class was predicted. Missing anatomical annotations are not negative GT.

In [ ]:
VIEW_AXIS: int = 2
MAX_SLICES: int = 24
ATTENTION_OPACITY: float = 0.6
SHOW_GT: bool = True
GT_REGION: int | None = 1

view_labels = None
if SHOW_GT:
    view_labels = patch_roi
figure = attention_figure(
    result=attention_result, labels=view_labels, region=GT_REGION,
    axis=VIEW_AXIS, max_slices=MAX_SLICES, opacity=ATTENTION_OPACITY,
)
figure.show(renderer="vscode")